# Pipeline Preprocessing Terintegrasi
## Geopolitics News × BI USD Rate (JISDOR)

Notebook ini mengimplementasikan pipeline preprocessing untuk menggabungkan dataset **berita geopolitik** (CNBC, kolom: `keyword_matched, title, url, preview_summary, full_text, date_raw, section`) dengan **data kurs USD Bank Indonesia** (`bi-usd-rate.csv`), untuk keperluan *time-series forecasting* / *sentiment-driven financial modeling*.

**Struktur pipeline:**
1. Setup & konfigurasi
2. Load data mentah
3. Filtering data berita geopolitik
4. Cleaning data kurs BI
5. Text preprocessing (NLP pipeline)
6. Feature engineering kurs BI (Kurs Tengah, log return)
7. Standardisasi zona waktu (WIB) & aturan *market cut-off* (T+1)
8. Penyelarasan hari libur/akhir pekan (*forward-rolling alignment*)
9. Agregasi berita harian (*group-by target date*)
10. Merge dataset final & ekspor


## 1. Setup & Instalasi Dependensi

In [27]:
# Jalankan sekali saja bila package belum tersedia di environment.
# Tanda seru (!) menjalankan perintah shell dari dalam notebook.
import sys
!{sys.executable} -m pip install -q --break-system-packages pandas numpy nltk Sastrawi langdetect || \
{sys.executable} -m pip install -q pandas numpy nltk Sastrawi langdetect



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import re
import html
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


## 2. Konfigurasi Pipeline

In [29]:
# Deteksi root proyek agar notebook tetap berjalan dari root proyek maupun folder src.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists() and (PROJECT_ROOT.parent / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NEWS_PATH = PROJECT_ROOT / "data" / "raw" / "2-cnbc-geoplotical-5years.csv"
BI_PATH   = PROJECT_ROOT / "data" / "raw" / "bi-usd-rate.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Parameter filtering berita
# ------------------------------------------------------------------
MIN_WORD_COUNT = 20                     # ambang batas minimal panjang teks
TITLE_REPEAT_WEIGHT = 2                 # bobot pengulangan judul saat penggabungan teks

RELEVANCE_KEYWORDS = [
    # 1. Istilah Inti Geopolitik
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    # 2. Konflik & Keamanan Militer
    "armed conflict",
    "global conflict",
    "international conflict2qwQQqqqwwwwwss",
    "military tensions",
    "national security",
    # 3. Geopolitik Ekonomi
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    # 4. Diplomasi & Kebijakan Luar Negeri
    "diplomacy",
    "foreign policy",
    # 5. Organisasi & Blok Internasional
    "NATO",
    "BRICS",
    "OPEC",
    "G7",
]


# Section/kategori yang dianggap TIDAK relevan (mis. hiburan, olahraga, gaya hidup)
# - daftar ini bersifat dapat dikonfigurasi (whitelist berita finansial/politik
#   terlalu ketat karena banyaknya nama rubrik CNBC yang beragam)
EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]
# HARUSNYA GAPERLU KARENA UDAH KITA PREPROCESSING SAAT SCRAPPING

# ------------------------------------------------------------------
# Parameter zona waktu & trading day
# ------------------------------------------------------------------
SOURCE_TZ = "America/New_York"   # asumsi: timestamp CNBC (`date_raw`) berbasis waktu newsroom AS (ET)
TARGET_TZ = "Asia/Jakarta"       # WIB (UTC+7)
MARKET_CUTOFF_HOUR = 15          # aturan T+1: berita > 15:00 WIB berdampak ke hari kerja berikutnya

print("Konfigurasi siap.")
print("NEWS_PATH:", NEWS_PATH)
print("BI_PATH:", BI_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

Konfigurasi siap.
NEWS_PATH: d:\nlpproject-grestercinta\data\raw\2-cnbc-geoplotical-5years.csv
BI_PATH: d:\nlpproject-grestercinta\data\raw\bi-usd-rate.csv
OUTPUT_DIR: d:\nlpproject-grestercinta\data\processed


## 3. Load Data Mentah

In [30]:
news_raw = pd.read_csv(NEWS_PATH)
bi_raw = pd.read_csv(BI_PATH, header=None) if False else pd.read_csv(BI_PATH)

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 7)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold


In [31]:
bi_raw.head(5)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal
0,1,1,17834.73,17657.27,9/1/2026 12:00:00 AM
1,2,1,17791.51,17614.49,8/31/2026 12:00:00 AM
2,3,1,17850.81,17673.19,8/28/2026 12:00:00 AM
3,4,1,17805.58,17628.42,8/27/2026 12:00:00 AM
4,5,1,17791.51,17614.49,8/26/2026 12:00:00 AM


## 4. Filtering Data Berita Geopolitik

Tahapan:
1. **Deduplikasi** berdasarkan `url`, lalu berdasarkan kombinasi `title + date_raw`.
2. **Filter teks kosong**: buang baris jika `full_text` **dan** `preview_summary` sama-sama null.
3. **Filter panjang teks**: buang berita dengan jumlah kata < `MIN_WORD_COUNT`.
4. **Filter relevansi kata kunci**: pastikan `keyword_matched` terisi atau teks memuat kata kunci geopolitik/ekonomi.
5. **Filter kategori**: buang section yang tidak relevan (`EXCLUDE_SECTIONS`). #ini harusnya gak bakal adasih pas scrapping.


### 4.1 Deduplicate Dataset

In [32]:
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Setelah di drop URL 
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "date_raw"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah dedup url                  : 16733
Setelah dedup title+date_raw       : 13965


### 4.2 Filter Teks Kosong

In [33]:
# Filter teks kosong: full_text DAN preview_summary null
mask_empty = news["full_text"].isna() & news["preview_summary"].isna()
news = news.loc[~mask_empty]
print(f"Setelah drop teks kosong             : {len(news)}")

Setelah drop teks kosong             : 13965


### 4.3 Filter Panjang Teks Berita <= 20

In [34]:
# Filter panjang teks minimal (gunakan full_text, fallback ke preview_summary)
effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= MIN_WORD_COUNT]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 13613


### 4.4 Filter Relefansi & Section Menggunakan Keyword

In [35]:
def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 13613
Setelah filter kategori/section     : 13573


,keyword_matched,title,url,preview_summary,full_text,date_raw,section
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets


## 5. Cleaning Data Kurs USD BI

Tahapan:
1. **Bersihkan metadata header/footer** — ambil hanya baris tabel riil dengan kolom `NO, Nilai, Kurs Jual, Kurs Beli, Tanggal` (fungsi dibuat generik agar tetap berfungsi meski file sumber memuat baris metadata di awal/akhir).
2. **Bersihkan karakter numerik** (pemisah ribuan) & konversi ke `float`.
3. **Filter outlier/anomali**: buang baris jika `Kurs Jual ≤ Kurs Beli`, atau bernilai nol/negatif.
4. Parse kolom `Tanggal` menjadi `datetime`.


In [36]:
EXPECTED_COLS = ["NO", "Nilai", "Kurs Jual", "Kurs Beli", "Tanggal"]

def clean_bi_header_footer(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Mengambil hanya blok tabel riil BI (kolom EXPECTED_COLS), membuang baris
    metadata/deskripsi yang mungkin ada di bagian atas atau bawah file mentah."""
    df = df_raw.copy()

    # Kasus 1: header sudah benar (kolom sesuai EXPECTED_COLS) -> tidak perlu diapa-apakan
    if list(df.columns[:5]) == EXPECTED_COLS:
        return df

    # Kasus 2: file memuat baris metadata di atas -> cari baris yang berisi header asli
    header_row_idx = None
    for i, row in df_raw.iterrows():
        row_vals = [str(v).strip() for v in row.values]
        if set(EXPECTED_COLS).issubset(set(row_vals)):
            header_row_idx = i
            break

    if header_row_idx is not None:
        new_header = df_raw.iloc[header_row_idx]
        df = df_raw.iloc[header_row_idx + 1:].copy()
        df.columns = new_header
        df = df.reset_index(drop=True)

    # Buang baris footer yang bukan data (mis. baris kosong / catatan sumber di akhir file)
    df = df.dropna(how="all")
    return df

bi = clean_bi_header_footer(bi_raw)
bi = bi[[c for c in EXPECTED_COLS if c in bi.columns]].copy()
print(bi.shape)
bi.head(3)


(1202, 5)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal
0,1,1,17834.73,17657.27,9/1/2026 12:00:00 AM
1,2,1,17791.51,17614.49,8/31/2026 12:00:00 AM
2,3,1,17850.81,17673.19,8/28/2026 12:00:00 AM


In [37]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Menghapus pemisah ribuan (koma) dan mengonversi ke float.
    Menangani baik format '17,834.73' maupun '17834.73'."""
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")

bi["Kurs Jual"] = clean_numeric(bi["Kurs Jual"])
bi["Kurs Beli"] = clean_numeric(bi["Kurs Beli"])
bi["NO"] = pd.to_numeric(bi["NO"], errors="coerce")
bi["Nilai"] = pd.to_numeric(bi["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi)}")

# Buang baris dengan nilai kurs nol/negatif atau NaN hasil konversi gagal
mask_valid_numeric = (
    bi["Kurs Jual"].notna() & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0) & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

# Validasi logis: Kurs Jual harus > Kurs Beli (spread positif)
mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

# Parse tanggal
bi["Tanggal"] = pd.to_datetime(bi["Tanggal"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
bi = bi.dropna(subset=["Tanggal"])
print(f"Setelah parsing tanggal valid : {len(bi)}")


bi = bi.sort_values("Tanggal").drop_duplicates(subset=["Tanggal"], keep="last").reset_index(drop=True)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)


Baris awal BI                 : 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah parsing tanggal valid : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03


## 6. Text Preprocessing (NLP Pipeline)

Tahapan:
1. **Penggabungan teks**: `title` (bobot 2x) + `preview_summary` + `full_text`.
2. **Normalization & cleaning**: hapus entitas HTML, tag, URL, email, simbol khusus, emoji.
3. **Lowercasing**.
4. **Deteksi bahasa** (EN/ID) — menentukan resource stopword & lemmatizer yang dipakai.
5. **Stopword removal** (Inggris via NLTK, Indonesia via daftar Sastrawi).
6. **Lemmatization/Stemming** — WordNet lemmatizer untuk EN, Sastrawi stemmer untuk ID.


In [38]:
URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)
NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s]")
MULTI_SPACE_RE = re.compile(r"\s+")

STOP_EN = set(stopwords.words("english"))
# Daftar stopword Bahasa Indonesia ringkas (dapat diperluas sesuai kebutuhan corpus)
STOP_ID = set("""
yang untuk pada ke para namun menurut antara dia dua ia seperti jika jika
sudah saat ini adalah kata dengan atau dari akan oleh dalam telah dapat
dan juga tersebut bahwa serta apa masih tidak tak setelah sekitar terhadap
lain kami itu lalu berikut namun tetapi bisa saja mungkin karena
""".split())

lemmatizer_en = WordNetLemmatizer()
stemmer_id = StemmerFactory().create_stemmer()

def detect_language(text: str) -> str:
    try:
        lang = detect(text[:500]) if isinstance(text, str) and text.strip() else "en"
        return "id" if lang == "id" else "en"
    except LangDetectException:
        return "en"

def basic_clean(text: str) -> str:
    """Normalisasi teks: hapus HTML/URL/email/simbol/emoji, lalu lowercase."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = NON_ALPHA_RE.sub(" ", t)   # buang angka & simbol khusus, sisakan huruf
    t = t.lower()
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t

def remove_stopwords_and_lemmatize(text: str, lang: str) -> str:
    tokens = [w for w in text.split() if len(w) > 2]
    if lang == "id":
        tokens = [w for w in tokens if w not in STOP_ID]
        tokens = [stemmer_id.stem(w) for w in tokens]
    else:
        tokens = [w for w in tokens if w not in STOP_EN]
        tokens = [lemmatizer_en.lemmatize(w) for w in tokens]
    return " ".join(tokens)

def preprocess_row(title, preview_summary, full_text):
    title = title if isinstance(title, str) else ""
    preview_summary = preview_summary if isinstance(preview_summary, str) else ""
    full_text = full_text if isinstance(full_text, str) else ""

    # Penggabungan dengan bobot pengulangan judul
    combined_raw = " ".join([title] * TITLE_REPEAT_WEIGHT + [preview_summary, full_text])

    lang = detect_language(combined_raw)
    cleaned = basic_clean(combined_raw)
    final_text = remove_stopwords_and_lemmatize(cleaned, lang)
    return pd.Series({"language": lang, "clean_text": final_text})

print("Fungsi preprocessing siap.")


Fungsi preprocessing siap.


In [39]:
# Terapkan pipeline NLP ke seluruh berita hasil filtering (Bagian 4)
result = news.apply(
    lambda row: preprocess_row(row["title"], row["preview_summary"], row["full_text"]),
    axis=1,
)
news = pd.concat([news.reset_index(drop=True), result.reset_index(drop=True)], axis=1)

# Buang baris yang setelah preprocessing menjadi kosong (mis. teks non-alfabet murni)
news = news.loc[news["clean_text"].str.len() > 0].reset_index(drop=True)

print(news.shape)
news[["title", "language", "clean_text"]].head(3)


(13573, 9)


,title,language,clean_text
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,en,global shipping authority warn maritime trade breakdown amidgeopoliticalturmoil global shipping authority warn marit...
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,en,morning call sheet rebound meet risinggeopoliticalrisks morning call sheet rebound meet risinggeopoliticalrisks pete...
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",en,global market keep shrugging shock could break streak according hsbc global market keep shrugging shock could break ...


## 7. Feature Engineering Kurs BI

- **Kurs Tengah** = (Kurs Jual + Kurs Beli) / 2
- **Return Logaritmik** $r_t = \ln(S_t / S_{t-1})$ — mengukur pergerakan harian nilai tukar.


In [40]:
bi["kurs_tengah"] = (bi["Kurs Jual"] + bi["Kurs Beli"]) / 2
bi["log_return"] = np.log(bi["kurs_tengah"] / bi["kurs_tengah"].shift(1))

bi[["Tanggal", "Kurs Jual", "Kurs Beli", "kurs_tengah", "log_return"]].head(5)


,Tanggal,Kurs Jual,Kurs Beli,kurs_tengah,log_return
0,2021-09-01,14377.53,14234.47,14306.000,NaN
1,2021-09-02,14355.42,14212.58,14284.000,-0.001539
2,2021-09-03,14352.41,14209.60,14281.005,-0.000210
3,2021-09-06,14332.31,14189.70,14261.005,-0.001401
4,2021-09-07,14310.20,14167.81,14239.005,-0.001544


## 8. Standardisasi Zona Waktu & Aturan Market Cut-off (T+1)

Langkah:
1. Konversi `date_raw` (asumsi zona waktu newsroom sumber: **`SOURCE_TZ`**) ke **WIB (`Asia/Jakarta`)**.
2. **Aturan T+1**: berita yang terbit setelah pukul `MARKET_CUTOFF_HOUR`:00 WIB dianggap berdampak ke hari kerja berikutnya, bukan hari yang sama.
3. **Forward-rolling alignment**: hasil tanggal dasar (setelah aturan cut-off) dipetakan ke *trading date* BI resmi berikutnya yang tersedia — ini otomatis menangani akhir pekan **dan** hari libur nasional, karena memakai himpunan tanggal perdagangan riil dari `bi-usd-rate.csv`, bukan kalender bisnis generik.

> **Catatan asumsi**: `date_raw` pada dataset berita tidak menyertakan info zona waktu eksplisit. Notebook ini mengasumsikan sumber berita (CNBC) mem-publish pada `SOURCE_TZ = "America/New_York"`; sesuaikan konfigurasi ini jika sumber data Anda berbeda.


In [41]:
# Himpunan tanggal perdagangan resmi BI (setelah cleaning)
trading_dates = np.array(sorted(bi["Tanggal"].dt.normalize().unique()))

def roll_forward_to_trading_date(base_date):
    """Memetakan base_date ke trading date BI berikutnya (>=). Mengembalikan NaT
    jika base_date melewati tanggal perdagangan terakhir yang tersedia."""
    idx = np.searchsorted(trading_dates, np.datetime64(base_date))
    if idx >= len(trading_dates):
        return pd.NaT
    return pd.Timestamp(trading_dates[idx])

# 8.1 Parse date_raw & localize ke SOURCE_TZ, lalu convert ke WIB
news["date_raw_dt"] = pd.to_datetime(news["date_raw"], errors="coerce")
news = news.dropna(subset=["date_raw_dt"]).reset_index(drop=True)

news["date_wib"] = (
    news["date_raw_dt"]
    .dt.tz_localize(SOURCE_TZ, ambiguous="NaT", nonexistent="NaT")
    .dt.tz_convert(TARGET_TZ)
)
news = news.dropna(subset=["date_wib"]).reset_index(drop=True)

# 8.2 Aturan T+1 market cut-off
is_after_cutoff = news["date_wib"].dt.hour >= MARKET_CUTOFF_HOUR
base_date = news["date_wib"].dt.normalize() + pd.to_timedelta(is_after_cutoff.astype(int), unit="D")
news["base_date"] = base_date.dt.tz_localize(None)

# 8.3 Forward-rolling alignment ke trading date BI resmi berikutnya
news["bi_trading_date"] = news["base_date"].apply(roll_forward_to_trading_date)

n_before_align = len(news)
news = news.dropna(subset=["bi_trading_date"]).reset_index(drop=True)
n_after_align = len(news)

print(f"Berita sebelum alignment tanggal : {n_before_align}")
print(f"Berita di luar cakupan data BI    : {n_before_align - n_after_align} (dibuang)")
print(f"Berita setelah alignment          : {n_after_align}")

news[["title", "date_raw", "date_wib", "base_date", "bi_trading_date"]].head(5)


Berita sebelum alignment tanggal : 13573
Berita di luar cakupan data BI    : 44 (dibuang)
Berita setelah alignment          : 13529


,title,date_raw,date_wib,base_date,bi_trading_date
0,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,7/31/2026 5:58:53 PM,2026-08-01 04:58:53+07:00,2026-08-01,2026-08-03
1,Central banks are bringing gold reserves home asgeopoliticalrisks rise,6/17/2026 2:28:36 PM,2026-06-18 01:28:36+07:00,2026-06-18,2026-06-18
2,Morning Call Sheet: AI optimism offsetsgeopoliticalmarket risks,6/16/2026 5:56:35 PM,2026-06-17 04:56:35+07:00,2026-06-17,2026-06-17
3,"Despite strong earnings, there’s elevated supply risks to Aramco given the attacks: CIO",8/5/2026 3:22:53 PM,2026-08-06 02:22:53+07:00,2026-08-06,2026-08-06
4,Staying overweight global stocks despitegeopoliticalrisks: Standard Chartered,6/2/2026 2:08:17 PM,2026-06-03 01:08:17+07:00,2026-06-03,2026-06-03


## 9. Agregasi Berita Harian (Group-by `bi_trading_date`)

Karena dalam satu hari perdagangan dapat terdapat puluhan–ratusan berita, seluruh teks yang jatuh pada `bi_trading_date` yang sama digabungkan (concatenated) menjadi satu representasi teks harian, sekaligus dihitung jumlah beritanya (`news_count`) sebagai fitur tambahan.


In [42]:
daily_news = (
    news.groupby("bi_trading_date")
    .agg(
        news_count=("clean_text", "size"),
        daily_text=("clean_text", lambda texts: " ".join(texts)),
        sections=("section", lambda s: list(pd.unique(s.dropna()))),
    )
    .reset_index()
    .rename(columns={"bi_trading_date": "Tanggal"})
)

print(daily_news.shape)
daily_news.head(5)


(1143, 4)


,Tanggal,news_count,daily_text,sections
0,2021-09-01,6605,fed rate hike biggerrisk chinageopoliticaltensions analyst fed rate hike biggerrisk chinageopoliticaltensions analys...,"[Street Signs Asia, Central Banks, Davos WEF, Market Insider, Politics, Europe News, Gold, The Rundown, Currencies, ..."
1,2021-09-02,1,jpmorgan top stock idea september jpmorgan top stock idea september jpmorgan released wednesday analyst focus list s...,[Pro: Street Calls]
2,2021-09-03,1,fintech war heating encroach turf investor fintech war heating encroach turf investor earlier week cnbc reported pay...,[Pro: Investing trends]
3,2021-09-06,1,inflation could repeat fed lost control niall ferguson say inflation could repeat fed lost control niall ferguson sa...,[Economy]
4,2021-09-08,2,guinea coup rattle iron ore bauxite market stokes economic uncertainty guinea coup rattle iron ore bauxite market st...,"[World Politics, Pro: Santoli on Stocks]"


## 10. Merge Dataset Final & Ekspor

Menggabungkan fitur kurs BI (`kurs_tengah`, `log_return`) dengan agregasi teks berita harian (`daily_text`, `news_count`) berdasarkan `Tanggal` (trading date).


In [43]:
final_df = bi.merge(daily_news, on="Tanggal", how="left")

# Hari perdagangan tanpa berita relevan -> isi default (tidak ada sinyal teks)
final_df["news_count"] = final_df["news_count"].fillna(0).astype(int)
final_df["daily_text"] = final_df["daily_text"].fillna("")
final_df["sections"] = final_df["sections"].apply(lambda x: x if isinstance(x, list) else [])

final_df = final_df.sort_values("Tanggal").reset_index(drop=True)

print(final_df.shape)
final_df[["Tanggal", "kurs_tengah", "log_return", "news_count"]].tail(10)


(1202, 10)


,Tanggal,kurs_tengah,log_return,news_count
1192,2026-08-18,17836.0,-0.002576,22
1193,2026-08-19,17856.0,0.001121,4
1194,2026-08-20,17844.0,-0.000672,4
1195,2026-08-21,17779.0,-0.003649,1
1196,2026-08-24,17705.0,-0.004171,11
1197,2026-08-26,17703.0,-0.000113,3
1198,2026-08-27,17717.0,0.000791,2
1199,2026-08-28,17762.0,0.002537,3
1200,2026-08-31,17703.0,-0.003327,12
1201,2026-09-01,17746.0,0.002426,9


In [44]:
# Ekspor hasil akhir
news_out_path = OUTPUT_DIR / "geopolitics_news_clean.csv"
bi_out_path = OUTPUT_DIR / "bi_usd_rate_clean.csv"
final_out_path = OUTPUT_DIR / "merged_news_fx_dataset.csv"

news.to_csv(news_out_path, index=False)
bi.to_csv(bi_out_path, index=False)
final_df.to_csv(final_out_path, index=False)

print("Tersimpan:")
print(" -", news_out_path)
print(" -", bi_out_path)
print(" -", final_out_path)


Tersimpan:
 - d:\nlpproject-grestercinta\data\processed\geopolitics_news_clean.csv
 - d:\nlpproject-grestercinta\data\processed\bi_usd_rate_clean.csv
 - d:\nlpproject-grestercinta\data\processed\merged_news_fx_dataset.csv


## Ringkasan Pipeline

| Tahap | Input | Output | Catatan |
|---|---|---|---|
| Filtering berita | `2-cnbc-geoplotical-5years.csv` | `news` (berita relevan) | dedup, panjang teks, relevansi, kategori |
| Cleaning kurs BI | `bi-usd-rate.csv` | `bi` (kurs bersih) | numerik, outlier, tanggal |
| NLP preprocessing | `news` | `clean_text`, `language` | normalisasi, stopword, lemma/stem |
| Feature engineering | `bi` | `kurs_tengah`, `log_return` | siap untuk pemodelan deret waktu |
| Alignment WIB & T+1 | `news`, `bi` | `bi_trading_date` | *forward-rolling* ke hari bursa BI riil |
| Agregasi harian | `news` | `daily_news` | gabung teks per hari perdagangan |
| Merge final | `bi`, `daily_news` | `final_df` | dataset siap model NLP + time-series |

**Parameter yang dapat disesuaikan** ada di **Bagian 2 (Konfigurasi)**: `MIN_WORD_COUNT`, `RELEVANCE_KEYWORDS`, `EXCLUDE_SECTIONS`, `SOURCE_TZ`, `MARKET_CUTOFF_HOUR`.
